# AI-powered Resume Screening System


In [ ]:
!pip install langchain==0.1.20 langchain-community==0.0.38 langsmith==0.1.17 transformers==4.39.3 requests==2.32.4

In [ ]:
!pip install langchain langchain-community transformers requests

In [ ]:
from langchain_core.prompts import PromptTemplate
from langchain.chains import LLMChain
from langchain_community.llms import HuggingFacePipeline

In [ ]:
import json
from langchain_core.runnables import RunnableSequence

In [ ]:
from transformers import pipeline
from langchain_community.llms import HuggingFacePipeline

hf_pipeline = pipeline(
    "text2text-generation",
    model="google/flan-t5-base",
    max_length=256,
    do_sample=True,
    temperature=0.3
)

llm = HuggingFacePipeline(pipeline=hf_pipeline)

In [ ]:
def safe_json(text):
    try:
        return json.loads(text)
    except:
        print("JSON issue:", text)
        return {}

In [ ]:
# Step 1: Skill Extraction
extract_prompt = PromptTemplate.from_template("""
You are helping in resume screening.

Extract the following from the resume:
- skills
- experience
- tools

Resume:
{resume}

Return ONLY JSON:
{{
  "skills": [],
  "experience": "",
  "tools": []
}}

Do NOT assume anything not written.
""")

In [ ]:
# Step 2: Matching
match_prompt = PromptTemplate.from_template("""
Compare candidate skills with job description.

Candidate Skills: {skills}
Job Description: {job}

Return JSON:
{{
  "matched_skills": [],
  "missing_skills": []
}}
""")


In [ ]:
# Step 3: Scoring
score_prompt = PromptTemplate.from_template("""
Give a score from 0 to 100.

Matched: {matched}
Missing: {missing}

Return JSON:
{{
  "score": 0
}}
""")

In [ ]:
# Step 4: Explanation
explain_prompt = PromptTemplate.from_template("""
Explain the score in simple words.

Score: {score}
Matched: {matched}
Missing: {missing}

Keep it short (2-3 lines).
""")

In [ ]:
# Step 1
extract_chain = extract_prompt | llm

# Step 2
match_chain = match_prompt | llm

# Step 3
score_chain = score_prompt | llm

# Step 4
explain_chain = explain_prompt | llm

In [ ]:
def simple_match(resume, job):
    resume = resume.lower()
    job = job.lower().split(",")

    matched = []
    missing = []

    for skill in job:
        skill = skill.strip()
        if skill in resume:
            matched.append(skill)
        else:
            missing.append(skill)

    return matched, missing

In [ ]:
def run_pipeline(resume, job):

    print("\n--- Step 1: Resume ---")
    print(resume)

    print("\n--- Step 2: Matching ---")
    matched, missing = simple_match(resume, job)
    print("Matched:", matched)
    print("Missing:", missing)

    print("\n--- Step 3: Scoring ---")
    if len(matched) + len(missing) == 0:
        score = 0
    else:
        score = int((len(matched) / (len(matched) + len(missing))) * 100)

    print("Score:", score)

    print("\n--- Step 4: Explanation ---")
    if score > 75:
        explanation = "Strong candidate with most required skills."
    elif score > 40:
        explanation = "Average candidate, some skills missing."
    else:
        explanation = "Weak candidate, lacks required skills."

    print(explanation)

    return {
        "matched": matched,
        "missing": missing,
        "score": score,
        "explanation": explanation
    }

In [ ]:
job_description = "python, machine learning, sql, cloud"

In [ ]:
resumes = {
    "Strong": "5 years experience in Python, Machine Learning, SQL, AWS cloud.",
    "Average": "Basic knowledge of Python and ML projects, no cloud.",
    "Weak": "Business graduate, no programming skills."
}

In [ ]:
for name, res in resumes.items():
    print(f"\n===== {name} =====")
    output = run_pipeline(res, job_description)

    print("FINAL SCORE:", output["score"])
    print("EXPLANATION:", output["explanation"])